# DIPCOMA Independent Testing — Clustering-Based Band Selection

**One-run Kaggle notebook.** This is the independent-test counterpart of the uploaded training notebook `clusteringbased(1).ipynb`. It preserves the two original methods: **K-means clustering** and **agglomerative hierarchical clustering (Ward linkage)**.

## Experimental protocol
1. Read training patches only from `dipcoma-trainval/dataset/train/`.
2. Fit the band-feature representation and clustering models on training patches only.
3. Freeze each selected original band subset. No test image is used to choose bands, fit clusters, or tune parameters.
4. Evaluate frozen selections on the six regions in `dipcoma-test`.
5. Report patch-level and region-level metrics, with region-level means giving each geographic region equal weight.

**Important data-layout correction:** DIPCOMA `.npy` cubes are stored as `(13, 512, 512)` = `(bands, height, width)`. This notebook explicitly validates and uses `BHW`; it does not guess the spectral axis.

The clustering notebook compares two independent clustering models, not the Group-Rank-Search hybrid. Ground-truth classification metrics are N/A because this dataset format does not provide aligned pixel-level class masks.


## 1. Kaggle setup

In **Add Input**, attach both datasets:
- `krrishshandilya/dipcoma-trainval`
- `krrishshandilya/dipcoma-test`

Then use **Run All**. The notebook searches `/kaggle/input` automatically. If the datasets are mounted under unusual names, set `TRAINVAL_ROOT` and `TEST_ROOT` in the configuration cell.


In [ ]:
import os, json, time, math, random, platform, warnings, zipfile, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy as sch
from scipy.spatial.distance import pdist
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from skimage.metrics import structural_similarity
from IPython.display import display, FileLink

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
print("Python:", platform.python_version(), "NumPy:", np.__version__, "Pandas:", pd.__version__)


In [ ]:
# ======================== CONFIGURATION ========================
CONFIG = {
    "trainval_root": None,          # normally auto-detected
    "test_root": None,              # normally auto-detected
    "bands_expected": 13,
    "train_patches_per_region": 5,  # sampled deterministically from each train region
    "train_pixels_per_patch": 2500,
    "test_patches_per_region": 20, # set None to evaluate all available patches (normally 66/region)
    "K_list": [3, 4, 5, 6, 8, 10, 12],
    "kmeans_seeds": [0, 1, 2, 3, 4],
    "kmeans_n_init": 10,
    "primary_linkage": "ward",
    "entropy_bins": 256,
    "ssim_downsample_step": 4,  # SSIM on 128x128 samples for speed; set 1 for full resolution
    "seed": 42,
    "output_dir": "/kaggle/working/testing_clustering_outputs",
}
TRAINVAL_ROOT = CONFIG["trainval_root"]
TEST_ROOT = CONFIG["test_root"]
OUT = Path(CONFIG["output_dir"]); OUT.mkdir(parents=True, exist_ok=True)
for sub in ["tables", "figures", "selections", "reports", "config"]: (OUT/sub).mkdir(exist_ok=True)
np.random.seed(CONFIG["seed"]); random.seed(CONFIG["seed"])


## 2. Dataset discovery and integrity checks

The training source is restricted to the `train` split of `dipcoma-trainval`. The independent test source is the separate `dipcoma-test` dataset. Only `.npy` patches with 13 spectral bands are accepted.


In [ ]:
def find_split_dir(split_name, preferred_token):
    root = Path("/kaggle/input")
    if not root.exists(): return None
    candidates = []
    for p in root.rglob(split_name):
        if p.is_dir() and any(p.rglob("*.npy")):
            score = (preferred_token in str(p).lower(), len(list(p.glob("*"))))
            candidates.append((score, p))
    if not candidates: return None
    candidates.sort(key=lambda x: (x[0][0], x[0][1]), reverse=True)
    return candidates[0][1]

def find_test_regions():
    root = Path("/kaggle/input")
    if not root.exists(): return None
    # Prefer a dataset path containing dipcoma-test; region directories are immediate children
    roots = [p for p in root.rglob("*") if p.is_dir() and "dipcoma-test" in str(p).lower()]
    roots = sorted(roots, key=lambda p: len(p.parts))
    for base in roots:
        for candidate in [base, base/"dataset", base/"test"]:
            if candidate.is_dir() and any(any(candidate.glob("*/"+"*.npy")) for _ in [0]):
                if len([d for d in candidate.iterdir() if d.is_dir() and list(d.glob("*.npy"))]) >= 2:
                    return candidate
    # Fallback: a directory with six or more immediate region subfolders containing npy patches
    for base in roots:
        for candidate in [base, base/"dataset", base/"test"]:
            if candidate.is_dir():
                n=sum(1 for d in candidate.iterdir() if d.is_dir() and any(d.glob("*.npy")))
                if n >= 2: return candidate
    return None

if TRAINVAL_ROOT is None:
    TRAIN_DIR = find_split_dir("train", "dipcoma-trainval")
else:
    tr=Path(TRAINVAL_ROOT); TRAIN_DIR = tr if tr.name.lower()=="train" else (tr/"dataset"/"train" if (tr/"dataset"/"train").exists() else tr/"train")
if TEST_ROOT is None:
    TEST_DIR = find_test_regions()
else:
    te=Path(TEST_ROOT); TEST_DIR = te if te.name.lower() in {"test","dataset"} else (te/"dataset" if (te/"dataset").exists() else te)
    if TEST_DIR.name.lower()=="dataset" and (TEST_DIR/"test").exists(): TEST_DIR=TEST_DIR/"test"
if TRAIN_DIR is None or not TRAIN_DIR.exists():
    raise FileNotFoundError("Training split not found. Attach dipcoma-trainval or set CONFIG['trainval_root'].")
if TEST_DIR is None or not TEST_DIR.exists():
    raise FileNotFoundError("Test regions not found. Attach dipcoma-test or set CONFIG['test_root'].")

train_regions={d.name: sorted(d.glob("*.npy")) for d in TRAIN_DIR.iterdir() if d.is_dir()}
test_regions={d.name: sorted(d.glob("*.npy")) for d in TEST_DIR.iterdir() if d.is_dir()}
train_regions={k:v for k,v in train_regions.items() if v}; test_regions={k:v for k,v in test_regions.items() if v}
print("TRAIN_DIR:", TRAIN_DIR, "| regions:", len(train_regions), "| patches:", sum(map(len,train_regions.values())))
print("TEST_DIR:", TEST_DIR, "| regions:", len(test_regions), "| patches:", sum(map(len,test_regions.values())))
print("Test regions:", {k:len(v) for k,v in sorted(test_regions.items())})
assert len(train_regions)>=2, "Expected multiple train-region directories."
assert len(test_regions)>=2, "Expected multiple independent test-region directories."


## 3. Load and validate DIPCOMA patches

Each patch must contain 13 bands. The source notebook's automatic axis heuristic can mistake a 13-band cube for a 512-band cube; here the expected layout is explicitly `(B,H,W)`. A last-axis 13-band cube is transposed to BHW.


In [ ]:
def load_bhw(path):
    a=np.load(path, mmap_mode="r")
    if a.ndim != 3: raise ValueError(f"{path} is not 3-D: {a.shape}")
    b=CONFIG["bands_expected"]
    if a.shape[0]==b: out=np.array(a, dtype=np.float32, copy=True)
    elif a.shape[-1]==b: out=np.asarray(np.moveaxis(a,-1,0), dtype=np.float32)
    else: raise ValueError(f"Expected {b} bands in first or last axis, got {a.shape} at {path}")
    if not np.isfinite(out).all():
        # Replace non-finite samples with per-band finite median; mark this in report
        for i in range(out.shape[0]):
            v=out[i]; good=np.isfinite(v)
            fill=float(np.median(v[good])) if good.any() else 0.0
            v[~good]=fill
    return out

def choose_files(region_map, limit, seed):
    rng=np.random.default_rng(seed); chosen={}
    for region, files in sorted(region_map.items()):
        fs=list(files)
        if limit is not None and len(fs)>limit:
            ix=np.sort(rng.choice(len(fs), size=limit, replace=False)); fs=[fs[i] for i in ix]
        chosen[region]=fs
    return chosen

TRAIN_FILES=choose_files(train_regions, CONFIG["train_patches_per_region"], CONFIG["seed"])
TEST_FILES=choose_files(test_regions, CONFIG["test_patches_per_region"], CONFIG["seed"]+1)
print("Training patches selected:", sum(map(len,TRAIN_FILES.values())))
print("Testing patches selected:", sum(map(len,TEST_FILES.values())))
# Verify common spatial dimensions and band count on representative files
for split_name, fmap in [("train",TRAIN_FILES),("test",TEST_FILES)]:
    shapes=set()
    for fs in fmap.values():
        if fs: shapes.add(load_bhw(fs[0]).shape)
    print(split_name, "sample BHW shapes:", sorted(shapes))
    assert all(s[0]==CONFIG["bands_expected"] for s in shapes)


## 4. Fit the shared band-feature space on training regions only

As in the source clustering notebook, each spectral band is represented by its values over sampled pixels. Rows are standardized and unit-normalized; Euclidean distance then corresponds to correlation distance: `||z_i-z_j||² = 2(1-r_ij)`. Sampling is balanced by region and patch to avoid one region dominating the fit.


In [ ]:
rng=np.random.default_rng(CONFIG["seed"]); train_samples=[]; train_region_sample_counts=[]
for region, files in TRAIN_FILES.items():
    n_before=sum(x.shape[1] for x in train_samples) if train_samples else 0
    for fp in files:
        cube=load_bhw(fp); B,H,W=cube.shape
        flat=cube.reshape(B,-1)
        valid=np.isfinite(flat).all(axis=0)
        ids=np.flatnonzero(valid)
        if len(ids)==0: continue
        n=min(CONFIG["train_pixels_per_patch"],len(ids))
        take=rng.choice(ids,size=n,replace=False)
        train_samples.append(flat[:,take])
    train_region_sample_counts.append({"region":region,"patches":len(files),"sampled_pixels":sum(x.shape[1] for x in train_samples)-n_before})
if not train_samples: raise RuntimeError("No usable training pixels were loaded.")
TRAIN_PIX=np.concatenate(train_samples,axis=1).astype(np.float64)
B=TRAIN_PIX.shape[0]; assert B==CONFIG["bands_expected"]
train_min=TRAIN_PIX.min(axis=1); train_max=TRAIN_PIX.max(axis=1); train_std=TRAIN_PIX.std(axis=1)
valid_bands=np.flatnonzero(train_std > 1e-8 * max(float(train_max.max()-train_min.min()),1.0))
if len(valid_bands)<3: raise RuntimeError(f"Only {len(valid_bands)} nonconstant bands in training data.")
Zraw=TRAIN_PIX[valid_bands]; mu=Zraw.mean(axis=1,keepdims=True); sd=Zraw.std(axis=1,keepdims=True)
Z=(Zraw-mu)/(sd*np.sqrt(Zraw.shape[1])); Z=Z.astype(np.float64)
R=np.clip(Z@Z.T,-1,1); D=pdist(Z,metric="euclidean")
assert np.isfinite(Z).all() and np.allclose(np.linalg.norm(Z,axis=1),1,atol=1e-6)
print("Training feature matrix:",Z.shape,"| original bands:",B,"| valid bands:",valid_bands.tolist())
print("Training sampled pixels:",TRAIN_PIX.shape[1],"| training data range:",float(train_max.max()-train_min.min()))
pd.DataFrame(train_region_sample_counts).to_csv(OUT/"tables"/"training_sampling.csv",index=False)


## 5. Clustering algorithms and representative-band selection

**Model A — K-means:** Lloyd K-means with k-means++ initialization. For each cluster, retain the original band nearest its cluster centroid.

**Model B — Agglomerative hierarchical clustering:** Ward linkage on the same Euclidean band-feature distances. Cut the tree into K clusters and retain the original band nearest each cluster centroid.

Both methods are fitted only on training data. K-means is repeated across the configured seeds; hierarchical Ward is deterministic.


In [ ]:
def representatives_nearest_centroid(Zm, labels, K):
    centers=np.vstack([Zm[labels==j].mean(axis=0) for j in range(K)])
    reps=[]
    for j in range(K):
        members=np.flatnonzero(labels==j)
        if not len(members): raise RuntimeError("Empty cluster encountered")
        reps.append(int(members[np.argmin(np.linalg.norm(Zm[members]-centers[j],axis=1))]))
    return np.asarray(reps,dtype=int), centers

def fit_kmeans(K, seed):
    t=time.perf_counter()
    model=KMeans(n_clusters=K,init="k-means++",n_init=CONFIG["kmeans_n_init"],max_iter=300,tol=1e-4,random_state=seed)
    labels=model.fit_predict(Z)
    reps,centers=representatives_nearest_centroid(Z,labels,K)
    return {"method":"KMeans","K":K,"seed":seed,"selected":np.sort(valid_bands[reps]),"labels":labels,"centers":centers,"inertia":float(model.inertia_),"fit_time_s":time.perf_counter()-t}

def fit_hierarchical(K):
    t=time.perf_counter(); linkage=sch.linkage(D,method=CONFIG["primary_linkage"]); labels=sch.cut_tree(linkage,n_clusters=K).ravel()
    reps,centers=representatives_nearest_centroid(Z,labels,K)
    return {"method":"Hierarchical_Ward","K":K,"seed":-1,"selected":np.sort(valid_bands[reps]),"labels":labels,"centers":centers,"linkage":linkage,"fit_time_s":time.perf_counter()-t}

K_LIST=sorted({int(k) for k in CONFIG["K_list"] if 2<=int(k)<len(valid_bands)})
if not K_LIST: raise ValueError("No valid K values; choose 2 <= K < number of valid bands.")
print("K values:",K_LIST)
FITTED=[]
for K in K_LIST:
    for seed in CONFIG["kmeans_seeds"]: FITTED.append(fit_kmeans(K,int(seed)))
    FITTED.append(fit_hierarchical(K))
selection_rows=[]
for m in FITTED:
    row={k:m[k] for k in ["method","K","seed","fit_time_s"]}; row["selected_bands_0based"]=json.dumps(m["selected"].tolist()); row["selected_bands_1based"]=json.dumps((m["selected"]+1).tolist()); row["inertia"]=m.get("inertia",np.nan); selection_rows.append(row)
    np.savez_compressed(OUT/"selections"/f"{m['method']}_K{m['K']}_seed{m['seed']}.npz",selected_bands_0based=m["selected"],labels=m["labels"],centers=m["centers"])
selection_df=pd.DataFrame(selection_rows); selection_df.to_csv(OUT/"tables"/"frozen_training_selections.csv",index=False)
display(selection_df)


## 6. Independent test metrics

For each held-out patch, reconstruct omitted bands by linear interpolation between the nearest selected original band indices (edge bands use the nearest selected band). Metrics are calculated in the original 13-band space. SSIM is evaluated on a regular spatial subsample controlled by `ssim_downsample_step` (default 4); set it to 1 for full-resolution SSIM if runtime permits. Compression ratio and bpp are raw selected-band payload estimates, consistent with the source clustering notebook; they are not measured JPEG2000 bitstream rates.


In [ ]:
def entropy_hist(values,bins=256):
    lo=float(np.min(values)); hi=float(np.max(values))
    if hi<=lo:return 0.0
    h,_=np.histogram(values,bins=bins,range=(lo,hi)); p=h[h>0]/h.sum()
    return float(-(p*np.log2(p)).sum())
TRAIN_ENT=np.array([entropy_hist(TRAIN_PIX[b],CONFIG["entropy_bins"]) for b in range(B)])
# Use the training-split extrema computed in the preceding cell. These bounds are frozen and reused for every independent test patch.
TRAIN_DR=max(float(train_max.max()-train_min.min()),1.0); REL_EPS=1e-4*TRAIN_DR

def reconstruct_cube(cube, selected):
    selected=np.sort(np.asarray(selected,dtype=int)); B,H,W=cube.shape
    rec=np.empty_like(cube,dtype=np.float32); rec[selected]=cube[selected]
    for b in range(B):
        if b in selected: continue
        r=int(np.searchsorted(selected,b))
        if r==0: rec[b]=cube[selected[0]]
        elif r==len(selected): rec[b]=cube[selected[-1]]
        else:
            left,right=selected[r-1],selected[r]; alpha=(b-left)/(right-left)
            rec[b]=(1-alpha)*cube[left]+alpha*cube[right]
    return rec

def patch_metrics(cube,selected):
    selected=np.sort(np.asarray(selected,dtype=int)); rec=reconstruct_cube(cube,selected)
    x=cube.astype(np.float64); y=rec.astype(np.float64); err=x-y
    mse=float(np.mean(err**2)); mae=float(np.mean(np.abs(err)))
    psnr=float("inf") if mse==0 else float(10*np.log10((TRAIN_DR**2)/mse))
    den=np.linalg.norm(x,axis=0)*np.linalg.norm(y,axis=0); ok=den>1e-12
    cos=np.ones_like(den); cos[ok]=np.sum(x[:,ok]*y[:,ok],axis=0)/den[ok]
    sam=float(np.degrees(np.arccos(np.clip(cos[ok],-1,1)).mean())) if ok.any() else np.nan
    omitted=np.setdiff1d(np.arange(B),selected)
    mre_vals=[]
    for b in omitted:
        mask=np.abs(x[b])>REL_EPS
        if mask.any(): mre_vals.append(float(np.mean(np.abs(err[b][mask])/np.abs(x[b][mask]))))
    step=int(CONFIG["ssim_downsample_step"]); ssim_vals=[]
    for b in omitted:
        a=x[b,::step,::step]; z=y[b,::step,::step]
        dr=float(max(a.max(),z.max())-min(a.min(),z.min()))
        if dr<=0: ssim_vals.append(1.0)
        else: ssim_vals.append(float(structural_similarity(a,z,data_range=dr,win_size=7)))
    if len(selected)>1:
        corr=np.corrcoef(cube[selected].reshape(len(selected),-1)); iu=np.triu_indices(len(selected),1); redundancy=float(np.nanmean(np.abs(corr[iu])))
    else: redundancy=np.nan
    return {"PSNR_full_dB":psnr,"SSIM_omitted":float(np.mean(ssim_vals)) if ssim_vals else np.nan,"SAM_deg":sam,"MAE_full":mae,"MRE_omitted":float(np.mean(mre_vals)) if mre_vals else np.nan,"redundancy_test":redundancy,"CR_raw":B/len(selected),"bpp_per_original_sample":len(selected)*16/B,"selected_bands":json.dumps(selected.tolist())}


## 7. Run independent evaluation on the six test regions

The default evaluates up to 20 patches per region (120 patches total), with a deterministic sample. Set `test_patches_per_region=None` to evaluate every available patch. Every fitted training selection is applied unchanged to every test patch. No clustering or band re-selection is performed on test data.


In [ ]:
test_rows=[]; t_eval=time.perf_counter()
for region, files in sorted(TEST_FILES.items()):
    print(f"Evaluating region {region}: {len(files)} patches")
    for fp in files:
        cube=load_bhw(fp)
        if cube.shape[0]!=B: raise ValueError(f"Band-count mismatch in {fp}: {cube.shape[0]} vs train {B}")
        for model in FITTED:
            met=patch_metrics(cube,model["selected"])
            test_rows.append({"region":region,"patch":fp.name,"method":model["method"],"K":model["K"],"seed":model["seed"],"selection_fit_time_s":model["fit_time_s"],**met})
    print("  completed")
patch_df=pd.DataFrame(test_rows)
patch_df.to_csv(OUT/"tables"/"test_patch_metrics.csv",index=False)
print("Patch-level evaluation rows:",len(patch_df),"elapsed seconds:",round(time.perf_counter()-t_eval,1))


In [ ]:
# Equal-region weighting: first average patches within each region, then average the region means.
metric_cols=["PSNR_full_dB","SSIM_omitted","SAM_deg","MAE_full","MRE_omitted","redundancy_test","CR_raw","bpp_per_original_sample","selection_fit_time_s"]
region_df=patch_df.groupby(["region","method","K","seed"],as_index=False)[metric_cols].mean()
region_df.to_csv(OUT/"tables"/"test_region_metrics.csv",index=False)
overall_df=region_df.groupby(["method","K"],as_index=False)[metric_cols].agg(["mean","std"])
overall_df.columns=["_".join([str(x) for x in c if x]).strip("_") if isinstance(c,tuple) else c for c in overall_df.columns]
overall_df.to_csv(OUT/"tables"/"test_overall_region_weighted.csv",index=False)
display(overall_df.round(5))


## 8. Figures and diagnostics

The figures below summarize the independent test performance. For stochastic K-means, results are shown across seeds; hierarchical clustering is deterministic. No single universal winner is declared because quality, compression and redundancy are competing objectives.


In [ ]:
# Compare region-weighted means across K.
plot_metrics=[("PSNR_full_dB_mean","PSNR (dB) — higher is better"),("SSIM_omitted_mean","SSIM — higher is better"),("SAM_deg_mean","SAM (degrees) — lower is better"),("MAE_full_mean","MAE — lower is better"),("redundancy_test_mean","Mean |correlation| — lower is better")]
fig,axs=plt.subplots(1,len(plot_metrics),figsize=(22,4))
for ax,(col,title) in zip(axs,plot_metrics):
    for method,g in overall_df.groupby("method"):
        ax.plot(g["K"],g[col],marker="o",label=method)
    ax.set_title(title,fontsize=9); ax.set_xlabel("Selected bands K"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT/"figures"/"test_metrics_vs_K.png",dpi=160,bbox_inches="tight"); plt.show()

# Region-level PSNR heatmap at the configured K nearest 6 (or median available K).
K_HEAT=6 if 6 in K_LIST else K_LIST[len(K_LIST)//2]
heat=region_df[region_df.K==K_HEAT].groupby(["region","method"])["PSNR_full_dB"].mean().unstack()
fig,ax=plt.subplots(figsize=(9,5)); im=ax.imshow(heat.values,aspect="auto",cmap="viridis"); ax.set_xticks(range(len(heat.columns))); ax.set_xticklabels(heat.columns,rotation=20); ax.set_yticks(range(len(heat.index))); ax.set_yticklabels(heat.index); ax.set_title(f"Independent test PSNR by region (K={K_HEAT})"); plt.colorbar(im,ax=ax,label="PSNR (dB)"); plt.tight_layout(); plt.savefig(OUT/"figures"/"region_psnr_heatmap.png",dpi=160,bbox_inches="tight"); plt.show()


## 9. Research interpretation and limitations

- The selected indices are learned from training-region patches only and frozen before testing.
- Region-weighted averages prevent regions with more patches from dominating the summary.
- K-means variability is summarized across seeds; hierarchical Ward has one deterministic result.
- `CR_raw` and `bpp_per_original_sample` describe raw selected-band payload size, not JPEG2000 bitstream compression. Do not compare these directly with codec-measured CR/bpp from the hybrid compression pipeline.
- SSIM is computed on spatially downsampled images when `ssim_downsample_step=4`; report this setting in the paper.
- No OA/AA/Kappa are reported because no aligned pixel-level class masks are used.
- The test patch cap is 20 per region by default. For final paper numbers, set it to `None` and rerun if compute budget permits.
- This notebook evaluates clustering-based selection only: K-means and hierarchical Ward. It does not include ranking, GA, BPSO or the hybrid.


In [ ]:
# Save configuration, summary, and a downloadable ZIP.
summary={
 "protocol":"fit on dipcoma-trainval train regions; evaluate frozen band subsets on dipcoma-test regions",
 "train_dir":str(TRAIN_DIR),"test_dir":str(TEST_DIR),
 "train_regions":{k:len(v) for k,v in TRAIN_FILES.items()},
 "test_regions":{k:len(v) for k,v in TEST_FILES.items()},
 "train_sampled_pixels":int(TRAIN_PIX.shape[1]),"bands":int(B),"valid_training_bands":valid_bands.tolist(),
 "K_list":K_LIST,"kmeans_seeds":CONFIG["kmeans_seeds"],"test_patch_count":int(patch_df[["region","patch"]].drop_duplicates().shape[0]),
 "test_evaluation_rows":int(len(patch_df)),"ssim_downsample_step":CONFIG["ssim_downsample_step"],
 "classification_metrics":"N/A (no aligned pixel-level labels)",
 "compression_metrics":"raw selected-band payload estimate; not JPEG2000 bitstream rate"
}
(OUT/"reports"/"run_summary.json").write_text(json.dumps(summary,indent=2))
(OUT/"config"/"config.json").write_text(json.dumps(CONFIG,indent=2))
(OUT/"reports"/"README.txt").write_text("DIPCOMA clustering independent-test evaluation. Models are fitted on trainval/train only and evaluated on dipcoma-test. See run_summary.json for exact counts and metric conventions.")
import shutil
zip_path=shutil.make_archive(str(OUT),"zip",root_dir=OUT.parent,base_dir=OUT.name)
print(json.dumps(summary,indent=2)); print("Results ZIP:",zip_path)
display(FileLink(zip_path))
